# 00. Getting Started — JupyterLab · Checking this PC · numpy basics

> This notebook is the warm-up for all the others. You learn how to use the tools, what is on this PC, and the
> **"photo = array of numbers"** idea (numpy) that every lesson uses.

### What you learn in this session
1. The JupyterLab screen and how to work with notebooks (cells · kernel · shortcuts)
2. Checking this PC: Python packages · AI devices (CPU/GPU/NPU) · the model folder
3. What a webcam photo looks like as numbers inside Python
4. Using numpy to crop a photo · swap the color order · build a model input shape

## 1. How to use JupyterLab

### The screen
- **Left**: the file list. Double-click a notebook (`.ipynb`) to open it. The folder shown here is your work folder `Documents\Edge Lab\notebooks` (the English notebooks are in its `en` folder).
- **Center**: the notebook. **Cells** (boxes) follow one after another from top to bottom.
- **Top right**: `Python 3 (ipykernel)` — the **kernel** you are connected to (the Python program that actually runs your code). A filled circle means it is busy running.

### Two kinds of cells
| Kind | What it does | Switch to it |
|---|---|---|
| Code cell | Python code. When you run it, the result appears right below | `Esc` → `Y` |
| Markdown cell | Explanation text (like this text) | `Esc` → `M` |

### Shortcuts you must know
| Key | Action |
|---|---|
| `Shift + Enter` | Run this cell and move to the next one |
| `Ctrl + Enter` | Run only this cell (stay where you are) |
| `Esc` / `Enter` | Command mode (select cells) / Edit mode (change text) |
| `A` / `B` (command mode) | New cell above / below |
| `D D` (command mode) | Delete the cell |
| `Ctrl + S` | Save |
| `I I` (command mode) | Stop a running cell |

### The kernel — variables carry over between cells
A variable you make in a cell above can be used in a cell below. That is why you must run the cells **in order, from the top**.
A number like `[3]` on the left of a cell shows the order it ran in. If the order gets mixed up or things act strange, use the menu **Kernel → Restart Kernel and Clear Outputs** and start again from the top.

### Saving and going back
Notebooks are saved in your work folder. To go back to the original (textbook) version, delete that notebook file and start `lab.bat` again — it only refills the missing files from the originals.

In [ ]:
# Let's run a cell — Shift + Enter
name = "edge-lab"
print("Hello,", name)
1 + 2          # if the last line of a cell is a value, you see the result even without print

In [ ]:
# The variable name from the cell above is still alive here (same kernel)
name.upper()

## 2. Notebook list — what you will learn

| No. | Topic | Key ideas | 🛠 Mini project |
|---|---|---|---|
| 00 | Getting started | JupyterLab, photo = array of numbers (numpy) | Photo filter booth |
| 01 | Finding faces · comparing devices | Basic OpenVINO flow, pre- and post-processing, speed · throughput | Face mosaic camera |
| 02 | Object detection by hand (ed-m) | Building the input, reading the output, writing IoU · NMS yourself, 2-step pose | People counter monitor |
| 03 | Face analysis | A pipeline that chains several models | Focus meter |
| 04 | Changing photos | Normalization, photo→photo models, alpha blending | Virtual background camera |
| 05 | Text · QR | Handling OCR results, reading order | Business card scanner |
| 06 | Hand · face points | Making rules from landmark coordinates | Finger rock-paper-scissors |
| 07 | Your own classifier | Transfer learning, softmax regression in numpy | Live classifier |
| 08 | Answering from documents (RAG) | Embeddings, cosine similarity, prompts | My-documents Q&A bot |
| 09 | Talking about photos (VLM) | Generation settings, streaming, JSON answers | Photo diary maker |
| 10 | Speech | Sound = numbers, speech-to-text, text-to-speech | Mini voice assistant |
| 11 | Board serial | Serial communication, a simple protocol | Sensor logger + remote LED |

Every notebook goes in this order: **concept → practice → common errors → summary → mini project**.
We suggest doing 01 → 02 → 03 in order. You can do the rest in any order you like.

## 3. Python and packages

What each package does:

| Package | What it does |
|---|---|
| `openvino` | Intel's inference engine. It loads models onto the CPU · GPU · NPU and runs them |
| `openvino-genai` | A bundle that makes text-generating models (LLM · VLM · Whisper) easy to use |
| `openvino-tokenizers` | Lets the text → numbers (tokens) step run as an OpenVINO model, too |
| `opencv-python` (`cv2`) | Reading photos · resizing · drawing · webcam |
| `numpy` | Math on arrays of numbers. In the end, a photo is a numpy array too |
| `onnxruntime` | Another engine that runs ONNX models (used for text-to-speech, TTS) |
| `mediapipe` | Google's hand · face point finder |
| `easyocr` | Reading text |
| `sounddevice` | Microphone recording · speakers |
| `pyserial` | Serial communication with a board |

The cell below **only checks what is installed** (it does not install anything). `setup.bat` does the installing.

In [ ]:
import sys
from importlib import metadata

print("python", sys.version.split()[0], "|", sys.executable)
for pkg in ["openvino", "openvino-genai", "openvino-tokenizers", "opencv-python", "numpy", "onnxruntime",
            "mediapipe", "easyocr", "sounddevice", "pyserial", "jupyterlab"]:
    try:
        print(f"  {pkg:22s} {metadata.version(pkg)}")      # only reads the installed version info
    except metadata.PackageNotFoundError:
        print(f"  {pkg:22s} (not installed)")

## 4. AI devices and the model folder

| Device | Good at | In these notebooks |
|---|---|---|
| **CPU** | Runs anything. Small models are fast enough. Pre- and post-processing always run on the CPU | Default, the baseline for comparison |
| **GPU** (built-in) | Big matrix math all at once. Big photos · big models | Object detection, changing photos, VLM |
| **NPU** | Steady work with little power. Strong with small models whose input size is **fixed** | Face analysis, feature extraction |

If `NPU` is not in the list, it is a driver problem — on Windows, look for **Neural processors** in Device Manager; on Ubuntu, use `linux-npu-driver`.

The model folder (`MODELS`) is `models\` inside the program folder. `lab.bat` tells the notebook where it is with the environment variable `EDGE_MODELS`.

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "../..")).resolve()       # program folder (lab.bat tells us)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # model folder
core = ov.Core()
print("Model folder:", MODELS)
print("Devices:", core.available_devices)

for d in core.available_devices:
    print(f"{d:6s}", core.get_property(d, "FULL_DEVICE_NAME"))

In [ ]:
# Models the notebooks use — if one is missing, only that notebook can't run
NEED = {
    "01 · 03 Face": "face/face-detection-retail-0005.xml",
    "02 Objects (ed-m)": "easydetect/ed-m/ed-m.onnx",
    "02 Pose (pose-m)": "easydetect/pose/pose-m.onnx",
    "03 Age·gender": "face/age-gender-recognition-retail-0013.xml",
    "03 Emotion": "face/emotions-recognition-retail-0003.xml",
    "03 Head pose": "face/head-pose-estimation-adas-0001.xml",
    "04 Remove background": "gan/u2net.onnx",
    "04 Upscale 4x": "gan/single-image-super-resolution-1032.xml",
    "04 Depth": "gan/depth-v2s.xml",
    "05 Text (easyocr)": "code/easyocr",
    "06 MediaPipe hands": "mediapipe/gesture_recognizer.task",
    "06 MediaPipe face": "mediapipe/face_landmarker.task",
    "07 Features": "backbone/mobilenetv2_feat.xml",
    "08 Embedding": "embed/e5-small-int8",
    "08 · 09 VLM": "vlm",
    "10 Speech-to-text": "stt",
    "10 Text-to-speech": "tts-int8",
}
for what, rel in NEED.items():
    print(f"{'OK     ' if (MODELS / rel).exists() else 'MISSING'}  {what:16s} models/{rel}")

## 5. Webcam — a photo is an array of numbers

`cv2.VideoCapture(0)` opens webcam number 0, and `read()` grabs one picture. The `frame` you get is a numpy array.
If there is no webcam, it uses `sample.jpg` in this notebook's folder (you can also put a different file name in `PHOTO`).

- On Windows, open it with `cv2.CAP_DSHOW` so it opens quickly.
- Right after a webcam turns on, the first few pictures come out dark, so we read 5 and use the last one.

In [ ]:
PHOTO = "sample.jpg"          # photo to use when there is no webcam (same folder as this notebook)

def show(bgr, width=720):
    """Show an OpenCV photo (BGR) inside the notebook."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """Grab one picture from the webcam. If there is no webcam, use the PHOTO file."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # some webcams give dark pictures for the first few frames
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"No webcam and no {PHOTO} either — put a photo file name in PHOTO"
        print("No webcam, so using the file:", PHOTO)
    return frame

frame = grab()
print("Photo size (height, width, channels):", frame.shape)
show(frame)

In [ ]:
# What is frame, really?
print("Type:", type(frame).__name__, "| shape (height, width, channels):", frame.shape, "| number type:", frame.dtype)
print("Value of the center pixel (B, G, R):", frame[frame.shape[0] // 2, frame.shape[1] // 2])
print("Smallest / largest value:", frame.min(), frame.max())

**How to read it**
- `(480, 640, 3)` = 480 rows tall × 640 columns wide × 3 colors. This order is called **HWC** (Height, Width, Channel).
- `uint8` = whole numbers from 0 to 255. 0 is dark and 255 is bright.
- OpenCV keeps colors in **B, G, R** order (the reverse of the usual R, G, B). Many models learned with RGB, so you often need to **swap** the order before feeding them.

## 6. numpy basics — everything you do before feeding a model
From now on, the "preprocessing" in every notebook is a mix of the five steps below.

In [ ]:
# ① Crop — [height range, width range]. The center 200 x 200
h, w = frame.shape[:2]
crop = frame[h // 2 - 100: h // 2 + 100, w // 2 - 100: w // 2 + 100]
print("Cropped:", crop.shape)

# ② Resize — cv2.resize takes (width, height)! (the shape is (height, width), so it is the other way round)
small = cv2.resize(frame, (320, 240))
print("Smaller:", small.shape)

# ③ Color order BGR → RGB — reverse the last axis (same as cv2.cvtColor)
rgb = frame[:, :, ::-1]
print("BGR first pixel:", frame[0, 0], "→ RGB first pixel:", rgb[0, 0])

# ④ Change the number type + scale to 0~1 — models usually take float32
f = frame.astype(np.float32) / 255.0
print("float32:", f.dtype, f.min(), "~", f.max())

# ⑤ Change the axis order + add a batch dimension — HWC → CHW → NCHW
chw = f.transpose(2, 0, 1)         # (channels, height, width)
nchw = chw[None]                   # add one 'how many photos' axis at the front → (1, channels, height, width)
print("HWC", f.shape, "→ CHW", chw.shape, "→ NCHW", nchw.shape)

**Summary** — If a model's input description says `[1, 3, 300, 300]`, it is **NCHW**, which means it wants the shape from ⑤.
If it says `[1, 224, 224, 3]`, it is **NHWC** — do not swap the axes, just add the batch axis at the front. The number range (0~255 / 0~1 / subtract the mean) and the color order are different for each model.

In [ ]:
# Try it yourself: combine the five steps above to make '300 x 300, BGR as is, 0~255, NCHW' (notebook 01 uses this shape)
x = cv2.resize(frame, (300, 300)).transpose(2, 0, 1)[None].astype(np.float32)
print(x.shape, x.dtype, x.min(), x.max())

## 7. Microphone · speakers

In [ ]:
try:
    import sounddevice as sd
    print(sd.query_devices())
    print("Default (input, output):", sd.default.device)
except (ImportError, OSError) as ex:           # Ubuntu needs PortAudio installed separately
    print("Could not read the sound devices:", ex)
    print("Ubuntu: sudo apt install libportaudio2  /  Windows: pip install -r requirements.lock.txt")

## 8. Watch out — memory
If edge-lab (`run.bat`) is running, its models are already in memory. If a notebook loads the same model again, you have **two copies**
(09 VLM alone is 3.5GB). For the big-model notebooks (08 · 09 · 10), close edge-lab first.

## Common errors
| Symptom | Cause · fix |
|---|---|
| `NameError: name 'frame' is not defined` | You did not run the cells above → run again from the top |
| `ModuleNotFoundError` | The package is missing → run `setup.bat` again |
| The webcam shows a black screen | Another program (edge-lab, a video call) is using the webcam |
| The cell never finishes (`[*]`) | Stop it with `I I` or Kernel → Interrupt |

## Summary
- Run notebooks from the top. Variables carry over between cells.
- A photo = a `(height, width, 3)` uint8 BGR array. Build the model input by matching the size · color order · number range · axis order.
- Devices: small fixed models on the NPU, big math on the GPU, everything else and pre/post-processing on the CPU.

---
# 🛠 Mini project — Photo filter booth

**What you build**: Take one webcam photo, apply four filters using only numpy (grayscale · invert · mosaic · sepia), show them side by side, and save the one you like to a file.
No AI here — this project is for getting comfortable with **photo = array of numbers**.

**What you use**: `grab` · `show` (above), numpy array math

| Filter | In numbers |
|---|---|
| Grayscale | Weighted average of the three colors: 0.114 B + 0.587 G + 0.299 R |
| Invert | 255 − value |
| Mosaic | Shrink it (averaging), then enlarge it again, keeping the blocks |
| Sepia | Mix the 3 colors with a 3 x 3 matrix |

In [ ]:
def gray(img):
    g = (img.astype(np.float32) @ np.array([0.114, 0.587, 0.299], np.float32)).astype(np.uint8)   # (H,W,3)@(3,) → (H,W)
    return np.stack([g, g, g], axis=-1)                       # back to 3 channels (so we can put them side by side)

def invert(img):
    return 255 - img

def mosaic(img, block=16):
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(1, w // block), max(1, h // block)), interpolation=cv2.INTER_AREA)   # shrink by averaging
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)                               # enlarge, keeping the blocks

SEPIA = np.array([[0.131, 0.534, 0.272],      # new B = old (B, G, R) mixed in these ratios
                  [0.168, 0.686, 0.349],      # new G
                  [0.189, 0.769, 0.393]], np.float32)   # new R
def sepia(img):
    return np.clip(img.astype(np.float32) @ SEPIA.T, 0, 255).astype(np.uint8)

shot = grab()
small = cv2.resize(shot, (320, 240))
board = np.vstack([np.hstack([small, gray(small), invert(small)]),
                   np.hstack([mosaic(small), sepia(small), np.zeros_like(small)])])
show(board, 960)

In [ ]:
choice = "sepia"                                   # one of gray · invert · mosaic · sepia
out = {"gray": gray, "invert": invert, "mosaic": mosaic, "sepia": sepia}[choice](shot)
cv2.imwrite(f"booth_{choice}.jpg", out)
print(f"Saved: booth_{choice}.jpg (you can see it in the file list on the left)")

**Try more**
- Mosaic only the faces → you do this in the notebook 01 mini project
- Brightness · contrast: `np.clip(img * contrast + brightness, 0, 255)`
- Mix the four filters to make your own filter